# bert-sms-spam-classification

In [1]:
import importlib.util
import subprocess
import sys

required = {
    "pandas": "pandas",
    "numpy": "numpy",
    "sklearn": "scikit-learn",
    "torch": "torch",
    "transformers": "transformers",
    "datasets": "datasets",
    "tqdm": "tqdm",
}

missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

print("All required packages are available.")

All required packages are available.


In [2]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)

def set_seed(seed=123):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(123)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


## 1. Load the SMS dataset

In [5]:
def standardize_spam_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    lower_map = {str(col).strip().lower(): col for col in df.columns}

    if {"class", "message"}.issubset(lower_map):
        df = df[[lower_map["class"], lower_map["message"]]].rename(
            columns={lower_map["class"]: "Class", lower_map["message"]: "Message"}
        )
    elif {"v1", "v2"}.issubset(lower_map):
        df = df[[lower_map["v1"], lower_map["v2"]]].rename(
            columns={lower_map["v1"]: "Class", lower_map["v2"]: "Message"}
        )
    elif {"label", "sms"}.issubset(lower_map):
        df = df[[lower_map["label"], lower_map["sms"]]].rename(
            columns={lower_map["label"]: "Class", lower_map["sms"]: "Message"}
        )
    elif df.shape[1] >= 2:
        first, second = df.columns[:2]
        df = df[[first, second]].rename(columns={first: "Class", second: "Message"})
    else:
        raise ValueError("Could not infer which columns contain the labels and messages.")

    df["Class"] = df["Class"].replace({0: "ham", 1: "spam", "0": "ham", "1": "spam"})
    df["Class"] = df["Class"].astype(str).str.strip().str.lower()
    df["Message"] = df["Message"].astype(str).str.strip()

    df = df[df["Class"].isin(["ham", "spam"])].copy()
    df = df[df["Message"].ne("")].copy()
    df["label"] = (df["Class"] == "spam").astype(int)

    return df[["Class", "Message", "label"]].reset_index(drop=True)


def read_csv_flexible(path: Path) -> pd.DataFrame:
    errors = []
    for encoding in ("utf-8", "latin-1"):
        try:
            return pd.read_csv(path, encoding=encoding)
        except Exception as exc:
            errors.append(f"{encoding}: {exc}")
    raise RuntimeError(f"Could not read {path}. Last error: {errors[-1]}")


def load_spam_sms_dataframe() -> pd.DataFrame:
    candidate_paths = [
        Path("Spam_SMS.csv"),
        Path("/mnt/data/Spam_SMS.csv"),
        Path("spam.csv"),
        Path("/mnt/data/spam.csv"),
        Path("SMSSpamCollection"),
        Path("/mnt/data/SMSSpamCollection"),
    ]

    for path in candidate_paths:
        if path.exists():
            if path.suffix.lower() == ".csv":
                raw_df = read_csv_flexible(path)
            else:
                raw_df = pd.read_csv(
                    path,
                    sep="\t",
                    header=None,
                    names=["Class", "Message"],
                    encoding="latin-1",
                )
            df = standardize_spam_dataframe(raw_df)
            print(f"Loaded dataset from: {path}")
            return df


    from datasets import load_dataset

    try:
        ds = load_dataset("ucirvine/sms_spam", "plain_text", split="train")
    except Exception:
        ds = load_dataset("ucirvine/sms_spam", split="train")

    raw_df = ds.to_pandas()
    df = standardize_spam_dataframe(raw_df)
    df[["Class", "Message"]].to_csv("Spam_SMS.csv", index=False)
    print("Loaded fallback dataset from Hugging Face and saved it locally as Spam_SMS.csv")
    return df


df = load_spam_sms_dataframe()
display(df.head())

print("Dataset shape:", df.shape)
print("\nClass distribution:")
display(df["Class"].value_counts().rename_axis("Class").reset_index(name="Count"))

Loaded dataset from: Spam_SMS.csv


,Class,Message,label
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


Dataset shape: (5574, 3)

Class distribution:


,Class,Count
0,ham,4827
1,spam,747


## 2. Split the dataset by required 80/20 train-test split

In [6]:
X = df["Message"]
y = df["label"]

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

# Internal validation split from the training portion only
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.10,
    random_state=42,
    stratify=y_train_full,
)

split_summary = pd.DataFrame(
    {
        "Split": ["Train", "Validation", "Test"],
        "Rows": [len(X_train), len(X_val), len(X_test)],
        "Spam": [int(y_train.sum()), int(y_val.sum()), int(y_test.sum())],
        "Ham": [int((1 - y_train).sum()), int((1 - y_val).sum()), int((1 - y_test).sum())],
    }
)

display(split_summary)
print(f"Required train/test split completed: {len(X_train_full)} training rows and {len(X_test)} test rows.")

,Split,Rows,Spam,Ham
0,Train,4013,538,3475
1,Validation,446,60,386
2,Test,1115,149,966


Required train/test split completed: 4459 training rows and 1115 test rows.


## 3. Tokenize the SMS text with `bert-base-uncased` and build PyTorch datasets

In [7]:
MODEL_NAME = "bert-base-uncased"
LOCAL_MODEL_DIR = Path("bert-base-uncased")
MODEL_SOURCE = str(LOCAL_MODEL_DIR) if LOCAL_MODEL_DIR.exists() else MODEL_NAME

MAX_LENGTH = 128
BATCH_SIZE = 16 if torch.cuda.is_available() else 8

CACHE_DIR = Path("hf_cache")
CACHE_DIR.mkdir(exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_SOURCE, use_fast=True, cache_dir=str(CACHE_DIR))

class SMSDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(list(texts), truncation=True, max_length=max_length)
        self.labels = [int(x) for x in labels]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: value[idx] for key, value in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item


train_dataset = SMSDataset(X_train.tolist(), y_train.tolist(), tokenizer, MAX_LENGTH)
val_dataset = SMSDataset(X_val.tolist(), y_val.tolist(), tokenizer, MAX_LENGTH)
test_dataset = SMSDataset(X_test.tolist(), y_test.tolist(), tokenizer, MAX_LENGTH)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=data_collator)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)

sample_batch = next(iter(train_loader))
print({key: tuple(value.shape) for key, value in sample_batch.items()})

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

E:\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tomhs\Harvard\Intro_LLM\Assignment 3\hf_cache\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'input_ids': (8, 65), 'token_type_ids': (8, 65), 'attention_mask': (8, 65), 'labels': (8,)}


## 4. Load BERT with a sequence-classification head and define training utilities

In [8]:
id2label = {0: "ham", 1: "spam"}
label2id = {"ham": 0, "spam": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_SOURCE,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    cache_dir=str(CACHE_DIR),
)
model.to(device)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=np.array(y_train.tolist()),
)
class_weights = torch.tensor(class_weights, dtype=torch.float, device=device)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

NUM_EPOCHS = 5
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

total_training_steps = len(train_loader) * NUM_EPOCHS
warmup_steps = int(0.10 * total_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps,
)

def compute_metrics_from_probs(y_true, probs, threshold=0.50):
    y_true = np.asarray(y_true).astype(int)
    probs = np.asarray(probs, dtype=float)
    preds = (probs >= threshold).astype(int)

    acc = accuracy_score(y_true, preds)
    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    f1 = 2 * precision * sensitivity / (precision + sensitivity) if (precision + sensitivity) else 0.0

    return {
        "accuracy": acc,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "precision": precision,
        "f1": f1,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "preds": preds,
    }


def select_best_threshold(y_true, probs):
    best_threshold = 0.50
    best_metrics = None
    best_score = (-1, -1, -1, -1)

    for threshold in np.linspace(0.05, 0.95, 181):
        metrics = compute_metrics_from_probs(y_true, probs, threshold)
        score = (
            metrics["accuracy"],
            metrics["sensitivity"],
            metrics["specificity"],
            metrics["f1"],
        )
        if score > best_score:
            best_score = score
            best_threshold = float(threshold)
            best_metrics = metrics

    return best_threshold, best_metrics


@torch.no_grad()
def predict_probabilities(model, data_loader):
    model.eval()
    all_probs, all_labels = [], []
    total_loss = 0.0

    for batch in data_loader:
        labels = batch["labels"].to(device)
        batch = {k: v.to(device) for k, v in batch.items() if k != "labels"}

        outputs = model(**batch)
        logits = outputs.logits
        loss = loss_fn(logits, labels)
        probs = torch.softmax(logits, dim=-1)[:, 1]

        total_loss += loss.item()
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return np.array(all_probs), np.array(all_labels), total_loss / len(data_loader)

print("Model and utilities are ready.")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model and utilities are ready.


## 5. Fine-tune the BERT model

In [9]:
best_model_path = Path("best_sms_spam_bert_state.pt")
history = []

best_val_score = (-1, -1, -1, -1)
best_threshold = 0.50

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss = 0.0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS}")
    for batch in progress_bar:
        labels = batch["labels"].to(device)
        batch = {k: v.to(device) for k, v in batch.items() if k != "labels"}

        optimizer.zero_grad(set_to_none=True)

        outputs = model(**batch)
        logits = outputs.logits
        loss = loss_fn(logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()
        progress_bar.set_postfix(batch_loss=f"{loss.item():.4f}")

    train_loss = running_loss / len(train_loader)

    val_probs, val_labels, val_loss = predict_probabilities(model, val_loader)
    candidate_threshold, val_metrics = select_best_threshold(val_labels, val_probs)

    current_score = (
        val_metrics["accuracy"],
        val_metrics["sensitivity"],
        val_metrics["specificity"],
        val_metrics["f1"],
    )

    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_accuracy": val_metrics["accuracy"],
            "val_sensitivity": val_metrics["sensitivity"],
            "val_specificity": val_metrics["specificity"],
            "val_f1": val_metrics["f1"],
            "selected_threshold": candidate_threshold,
        }
    )

    print(
        f"Epoch {epoch}: "
        f"train_loss={train_loss:.4f}, "
        f"val_loss={val_loss:.4f}, "
        f"val_accuracy={val_metrics['accuracy']:.4f}, "
        f"val_sensitivity={val_metrics['sensitivity']:.4f}, "
        f"val_specificity={val_metrics['specificity']:.4f}, "
        f"threshold={candidate_threshold:.3f}"
    )

    if current_score > best_val_score:
        best_val_score = current_score
        best_threshold = candidate_threshold
        torch.save(model.state_dict(), best_model_path)

history_df = pd.DataFrame(history)
display(history_df)

print(f"\nBest validation threshold  : {best_threshold:.3f}")
print(f"Best validation accuracy   : {best_val_score[0]:.4f}")
print(f"Best validation sensitivity: {best_val_score[1]:.4f}")
print(f"Best validation specificity: {best_val_score[2]:.4f}")

Epoch 1/5:   0%|          | 0/502 [00:00<?, ?it/s]

Epoch 1: train_loss=0.2258, val_loss=0.1892, val_accuracy=0.9888, val_sensitivity=0.9167, val_specificity=1.0000, threshold=0.050


Epoch 2/5:   0%|          | 0/502 [00:00<?, ?it/s]

Epoch 2: train_loss=0.0624, val_loss=0.1212, val_accuracy=0.9933, val_sensitivity=0.9500, val_specificity=1.0000, threshold=0.050


Epoch 3/5:   0%|          | 0/502 [00:00<?, ?it/s]

...

Epoch 4: train_loss=0.0129, val_loss=0.1223, val_accuracy=0.9910, val_sensitivity=0.9667, val_specificity=0.9948, threshold=0.710


Epoch 5/5:   0%|          | 0/502 [00:00<?, ?it/s]

Epoch 5: train_loss=0.0042, val_loss=0.1029, val_accuracy=0.9933, val_sensitivity=0.9667, val_specificity=0.9974, threshold=0.825


,epoch,train_loss,val_loss,val_accuracy,val_sensitivity,val_specificity,val_f1,selected_threshold
0,1,0.225761,0.189224,0.988789,0.916667,1.000000,0.956522,0.050
1,2,0.062364,0.121207,0.993274,0.950000,1.000000,0.974359,0.050
2,3,0.011584,0.089523,0.993274,0.966667,0.997409,0.974790,0.050
3,4,0.012926,0.122264,0.991031,0.966667,0.994819,0.966667,0.710
4,5,0.004159,0.102873,0.993274,0.966667,0.997409,0.974790,0.825



Best validation threshold  : 0.050
Best validation accuracy   : 0.9933
Best validation sensitivity: 0.9667
Best validation specificity: 0.9974


## 6. Evaluate on the untouched 20% test split

In [10]:
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.to(device)

test_probs, test_labels, test_loss = predict_probabilities(model, test_loader)
test_metrics = compute_metrics_from_probs(test_labels, test_probs, threshold=best_threshold)

results_df = pd.DataFrame(
    {
        "Metric": [
            "Test loss",
            "Test accuracy",
            "Test sensitivity (spam recall)",
            "Test specificity (ham recall)",
            "Decision threshold",
        ],
        "Value": [
            test_loss,
            test_metrics["accuracy"],
            test_metrics["sensitivity"],
            test_metrics["specificity"],
            best_threshold,
        ],
    }
)

display(results_df)

confusion_df = pd.DataFrame(
    [
        [test_metrics["tn"], test_metrics["fp"]],
        [test_metrics["fn"], test_metrics["tp"]],
    ],
    index=["Actual ham", "Actual spam"],
    columns=["Predicted ham", "Predicted spam"],
)

display(confusion_df)

print(
    classification_report(
        test_labels,
        test_metrics["preds"],
        target_names=["ham", "spam"],
        digits=4,
    )
)

if test_metrics["accuracy"] >= 0.99:
    print(f"Target met: test accuracy = {test_metrics['accuracy']:.4f} (>= 0.99).")
else:
    print(f"Target not met in this run: test accuracy = {test_metrics['accuracy']:.4f}.")

,Metric,Value
0,Test loss,0.139653
1,Test accuracy,0.991928
2,Test sensitivity (spam recall),0.966443
3,Test specificity (ham recall),0.995859
4,Decision threshold,0.050000


,Predicted ham,Predicted spam
Actual ham,962,4
Actual spam,5,144


              precision    recall  f1-score   support

         ham     0.9948    0.9959    0.9953       966
        spam     0.9730    0.9664    0.9697       149

    accuracy                         0.9919      1115
   macro avg     0.9839    0.9812    0.9825      1115
weighted avg     0.9919    0.9919    0.9919      1115

Target met: test accuracy = 0.9919 (>= 0.99).
